# Gemma 4 Hybrid Pipeline (YOLO detects, Gemma 4 explains) - Kaggle

In [ ]:
# 1. Check GPU - verify CUDA is available and print GPU name and VRAM
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Install packages - install all required Python libraries
!pip install -q torch accelerate
!pip install -q "transformers>=5.10.1" ultralytics pillow

In [ ]:
# 4. Imports - load all Python libraries used across this notebook
import os, json, re, time, tempfile
from pathlib import Path
from PIL import Image
import torch
import transformers


In [ ]:
# Load HuggingFace token - read API token from Kaggle secrets
import os, transformers
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HUGGINGFACE_TOKEN')
os.environ['HF_TOKEN'] = hf_token
transformers.logging.set_verbosity_error()
print('Token loaded.')

In [ ]:
# ── Config + robust path resolution ──────────────────────────────────
import glob
from pathlib import Path

IMAGES_DIR      = '/kaggle/input/rq2-explanation-sample-flat'
YOLO_MODEL_PATH = '/kaggle/input/yolo11s-visdrone-finetuned/fine-tuned.pt'

MODEL_ID = 'google/gemma-4-E2B-it'

MAX_IMAGES         = 0     # 0 = all images in the sample
MAX_DETS_PER_IMAGE = 30    # top-N YOLO boxes per image by confidence
MIN_CONFIDENCE     = 0.40  # ignore low-confidence YOLO boxes
OUTPUT_DIR         = '/kaggle/working'

def _first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

if not Path(YOLO_MODEL_PATH).is_file():
    found = _first('/kaggle/input/**/fine-tuned.pt') or _first('/kaggle/input/**/*.pt')
    if found:
        YOLO_MODEL_PATH = found
        print('Discovered YOLO weights:', YOLO_MODEL_PATH)

if not (Path(IMAGES_DIR).is_dir() and list(Path(IMAGES_DIR).glob('*.jpg'))):
    jpgs = glob.glob('/kaggle/input/**/*.jpg', recursive=True)
    if jpgs:
        IMAGES_DIR = str(Path(jpgs[0]).parent)
        print('Discovered images dir:', IMAGES_DIR)

n_imgs = len(list(Path(IMAGES_DIR).glob('*.jpg')))
print('Images dir exists :', Path(IMAGES_DIR).is_dir())
print('YOLO weights exist:', Path(YOLO_MODEL_PATH).is_file())
print('Images found      :', n_imgs)
assert n_imgs > 0,                      'No images found - attach dataset / fix IMAGES_DIR.'
assert Path(YOLO_MODEL_PATH).is_file(), 'YOLO weights not found - attach dataset / fix YOLO_MODEL_PATH.'

In [ ]:
# 7. Load YOLO model - load the fine-tuned YOLO detector from Drive
from ultralytics import YOLO

yolo = YOLO(YOLO_MODEL_PATH)
print('YOLO loaded.')

In [ ]:
# 8. Load model - download and initialise Gemma 4 pipeline on GPU
# pipeline() is Google's recommended approach for Gemma 4 vision tasks
from transformers import pipeline, GenerationConfig

vqa_pipe = pipeline(
    task='image-text-to-text',
    model=MODEL_ID,
    device_map='auto',
    torch_dtype=torch.float16,
    token=hf_token,
)

gen_config = GenerationConfig.from_pretrained(MODEL_ID)
gen_config.max_new_tokens = 1024
gen_kwargs = dict(generation_config=gen_config)

print('Gemma 4 loaded. Memory used:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')


In [ ]:
# 9. Define helper functions - explanation prompt, inference runner, and bbox crop
import json, re, tempfile
from pathlib import Path
from PIL import Image, ImageDraw


def highlight_detection(image_path, bbox, color='red', width=3):
    """Draw a single bounding box highlight on the image and return it."""
    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    x1, y1, x2, y2 = [int(v) for v in bbox]
    for offset in range(width):
        draw.rectangle([x1 - offset, y1 - offset, x2 + offset, y2 + offset], outline=color)
    return img


def build_explanation_prompt(class_name, bbox, img_w, img_h):
    """Build the prompt asking the VLM to explain a specific YOLO detection."""
    x1, y1, x2, y2 = [int(v) for v in bbox]
    cx_pct = round(((x1 + x2) / 2) / img_w * 100)
    cy_pct = round(((y1 + y2) / 2) / img_h * 100)
    obj_w  = round((x2 - x1) / img_w * 100)
    obj_h  = round((y2 - y1) / img_h * 100)

    return (
        f'This is an aerial drone image taken from above. '
        f'An object detector has identified a {class_name} and it is highlighted '
        f'with a red bounding box. The object is located at approximately '
        f'{cx_pct}% from the left and {cy_pct}% from the top of the image, '
        f'occupying about {obj_w}% of the image width and {obj_h}% of the image height.\n\n'
        f'In one sentence, describe this {class_name} as you see it -- '
        f'its colour, what it looks like, and what it is doing.'
        f''
    )


def get_vlm_explanation(pil_image, prompt_text):
    """Run a single Gemma 4 inference call and return the response string."""
    # Pipeline expects a file path; save to a temp file and delete it after the call
    tmp = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False, dir='/tmp')
    tmp.close()
    pil_image.convert('RGB').save(tmp.name, format='JPEG', quality=95)

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'url': tmp.name},
            {'type': 'text',  'text': prompt_text},
        ]
    }]

    output = vqa_pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
    os.unlink(tmp.name)

    response = re.sub(r'<\s*/?\s*(?:end_of_)?turn[^>]*>', '', output[0]['generated_text']).strip()
    torch.cuda.empty_cache()
    return response


print('Helper functions defined.')

In [ ]:
# 10. Smoke test - run on 2 images to verify the pipeline before the full run
from pathlib import Path

image_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))[:2]

for image_path in image_paths:
    print(f'\n--- {image_path.name} ---')

    results = yolo.predict(source=str(image_path), conf=MIN_CONFIDENCE,
                           save=False, verbose=False)
    boxes = results[0].boxes
    print(f'YOLO found {len(boxes)} detections')

    if len(boxes) == 0:
        continue

    top_box    = sorted(boxes, key=lambda b: float(b.conf[0]), reverse=True)[0]
    class_name = yolo.names[int(top_box.cls[0])]
    bbox       = top_box.xyxy[0].tolist()
    confidence = float(top_box.conf[0])

    img         = Image.open(image_path).convert('RGB')
    highlighted = highlight_detection(image_path, bbox)
    prompt      = build_explanation_prompt(class_name, bbox, img.width, img.height)

    print(f'Explaining: {class_name} (conf={confidence:.2f})')
    print(f'Prompt: {prompt[:120]}...')

    explanation = get_vlm_explanation(highlighted, prompt)
    print(f'Explanation: {explanation}')

print('\nSmoke test complete.')

In [ ]:
# 11. Full run - run YOLO detection then Gemma explanation on all images
import json, time
from pathlib import Path

all_image_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))
if MAX_IMAGES and MAX_IMAGES > 0:
    all_image_paths = all_image_paths[:MAX_IMAGES]

cap_str = str(MAX_DETS_PER_IMAGE) if MAX_DETS_PER_IMAGE else 'no limit'
print(f'Processing {len(all_image_paths)} images, {cap_str} detections each.\n')

results_all = {}
predictions = {}
timing_log  = {}
run_start   = time.perf_counter()

for i, image_path in enumerate(all_image_paths):
    img_start = time.perf_counter()
    print(f'[{i+1}/{len(all_image_paths)}] {image_path.name}', end=' ... ', flush=True)

    yolo_results = yolo.predict(source=str(image_path), conf=MIN_CONFIDENCE,
                                save=False, verbose=False)
    boxes = yolo_results[0].boxes

    if len(boxes) == 0:
        print('no detections, skipping')
        results_all[image_path.name] = []
        predictions[image_path.name] = []
        timing_log[image_path.name]  = {'total_s': 0, 'num_detections': 0, 'per_det_s': []}
        continue

    sorted_boxes = sorted(boxes, key=lambda b: float(b.conf[0]), reverse=True)
    selected     = sorted_boxes if not MAX_DETS_PER_IMAGE else sorted_boxes[:MAX_DETS_PER_IMAGE]

    img      = Image.open(image_path).convert('RGB')
    img_w, img_h = img.size

    image_results = []
    image_preds   = []
    per_det_times = []

    for box in selected:
        class_id   = int(box.cls[0])
        class_name = yolo.names[class_id]
        confidence = float(box.conf[0])
        bbox       = box.xyxy[0].tolist()

        det_start   = time.perf_counter()
        highlighted = highlight_detection(image_path, bbox)
        prompt      = build_explanation_prompt(class_name, bbox, img_w, img_h)
        explanation = get_vlm_explanation(highlighted, prompt)
        per_det_times.append(round(time.perf_counter() - det_start, 2))

        image_results.append({
            'class_id':    class_id,
            'class_name':  class_name,
            'confidence':  confidence,
            'bbox':        bbox,
            'explanation': explanation,
        })
        image_preds.append({
            'class_id':   class_id,
            'class_name': class_name,
            'confidence': confidence,
            'bbox':       bbox,
        })

    img_elapsed = time.perf_counter() - img_start
    timing_log[image_path.name] = {
        'total_s':        round(img_elapsed, 2),
        'num_detections': len(image_results),
        'avg_per_det_s':  round(sum(per_det_times) / len(per_det_times), 2) if per_det_times else 0,
        'per_det_s':      per_det_times,
    }

    results_all[image_path.name] = image_results
    predictions[image_path.name] = image_preds

    elapsed = time.perf_counter() - run_start
    avg_per_img = elapsed / (i + 1)
    eta = avg_per_img * (len(all_image_paths) - i - 1)
    print(f'{len(image_results)} explained [{img_elapsed:.1f}s | avg {avg_per_img:.1f}s/img | ETA {eta/60:.0f}min]')

    if (i + 1) % 10 == 0:
        with open(f'{OUTPUT_DIR}/gemma4_hybrid_partial.json', 'w') as f:
            json.dump(results_all, f, indent=2)

total_elapsed = time.perf_counter() - run_start
all_det_times = [t for v in timing_log.values() for t in v.get('per_det_s', [])]
print(f'\nDone. Total: {total_elapsed/60:.1f} min | Avg per det: {sum(all_det_times)/len(all_det_times):.1f}s')


In [ ]:
# 12. Save results - write explanations, predictions, and timing JSON to Drive
import json

# Full output with explanations (for RQ2 analysis and visualize_explanations.py)
explanations_path = f'{OUTPUT_DIR}/gemma4_hybrid_explanations.json'
with open(explanations_path, 'w') as f:
    json.dump({
        'model':   MODEL_ID,
        'setting': 'hybrid',
        'config':  {
            'max_images':         MAX_IMAGES,
            'max_dets_per_image': MAX_DETS_PER_IMAGE,
            'min_confidence':     MIN_CONFIDENCE,
        },
        'images': results_all,
    }, f, indent=2)

# Standard predictions format (for evaluate.py if needed)
predictions_path = f'{OUTPUT_DIR}/gemma4_hybrid_predictions.json'
with open(predictions_path, 'w') as f:
    json.dump(predictions, f, indent=2)

total_explained = sum(len(v) for v in results_all.values())
print(f'Saved {len(results_all)} images, {total_explained} explanations')
print(f'Explanations: {explanations_path}')
print(f'Predictions:  {predictions_path}')
print('\nDownload both files from the Output tab ->')

# Timing
timing_path = f'{OUTPUT_DIR}/gemma4_hybrid_timing.json'
with open(timing_path, 'w') as f:
    json.dump({
        'model':     MODEL_ID,
        'setting':   'hybrid',
        'per_image': timing_log,
    }, f, indent=2)
print(f'Timing:       {timing_path}')
